In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")

In [5]:
culvercheck_df = pd.read_csv(f"{GCS_FILE_PATH}/Ridership by Time Period, Route, and Stop_7_14_25-8_25_25.csv")

In [6]:
bart_ridership.sample(5)

,FY,Year,Month,Month Name,Date,Day of Week,Day Type,Station,Entries,Exits,Station Name,start_date,end_date
6588,FY25,2025,2,Feb,2025-02-09,Sunday,Sunday,RM,1077,1022,Richmond,2025-02-09,2025-02-09
1001,FY25,2024,10,Oct,2024-10-21,Monday,Weekday,16,5310,5245,16th Street Mission,2024-10-21,2024-10-21
9457,FY25,2025,4,Apr,2025-04-08,Tuesday,Weekday,BF,2898,2953,Bayfair,2025-04-08,2025-04-08
4776,FY25,2025,1,Jan,2025-01-04,Saturday,Saturday,MA,1796,1838,MacArthur,2025-01-04,2025-01-04
3172,FY25,2024,12,Dec,2024-12-03,Tuesday,Weekday,GP,3440,3416,Glen Park,2024-12-03,2024-12-03


In [7]:
big_blue_bus_ridership.sample(5)

,SERVICE_PERIOD,SERVICE_DAY,ROUTE_NUMBER,ROUTE_NAME,DIRECTION_NAME,STOP_ID,STOP_NAME,STOP_LAT,STOP_LON,AVERAGE_DAILY_BOARDINGS,AVERAGE_DAILY_ALIGHTINGS,start_date,end_date
6310,2025-04-01,SATURDAY,18,UCLA - Abbot Kinney,WESTBOUND,2219,MONTANA WB/SAN VICENTE FS,34.052330,-118.470513,9.508333,6.627778,2025-04-01,2025-07-31
3697,2024-12-01,SUNDAY,2,Wilshire Blvd/UCLA,WESTBOUND,1070,4TH SB/SANTA MONICA FS,34.016191,-118.495041,2.227381,27.286904,2024-12-01,2025-03-31
9979,2025-08-01,WEEKDAY,1,Main St & Santa Monica Blvd/UCLA,EASTBOUND,2155,WESTWOOD NB/ROCHESTER NS,34.056071,-118.442095,33.486405,50.390099,2025-08-01,2025-11-30
8126,2025-04-01,WEEKDAY,27,Pico Blvd Rapid,EASTBOUND,2028,PICO EB/BUNDY NS,34.029146,-118.450481,12.445058,14.090685,2025-04-01,2025-07-31
7269,2025-04-01,WEEKDAY,3,Lincoln Blvd/LAX,SOUTHBOUND,2695,LINCOLN SB/SUPERBA NS,33.996382,-118.457834,13.075811,9.717872,2025-04-01,2025-07-31


In [8]:
caltrain_ridership.sample(5)

,Month,Origin Station,Caltrain Ridership,Date Type,Average Ridership,start_date,end_date
1136,March 2024,Santa Clara,14118.114380,Saturday,349.390189,2024-03-01,2024-03-31
956,September 2024,Santa Clara,18382.195390,Saturday,663.322392,2024-09-01,2024-09-30
223,December 2024,Menlo Park,17657.013906,Weekday,717.948511,2024-12-01,2024-12-31
2296,June 2024,Mountain View,42028.667563,Holiday,0.000000,2024-06-01,2024-06-30
920,October 2024,San Bruno,8725.544916,Saturday,211.594386,2024-10-01,2024-10-31


In [9]:
culver_citybus_ridership.sample(5)

,Day Of Week,Route,Direction,Stop ID,Stop Name,AVG On,AVG Off,start_date,end_date,route_id,day_type
1156,3-Sunday,6-Sepulveda Boulevard,outbound,693,LAX/METRO TRANSIT CENTER BAY,0.0,460.1,2025-07-14,2025-08-25,6,Sunday
891,3-Sunday,1-Washington Boulevard,Inbound,125,Washington Blvd/Girard Ave,7.7,8.0,2025-07-14,2025-08-25,1,Sunday
1021,3-Sunday,3-Crosstown,Inbound,416,Jefferson Blvd/Machado Rd,3.8,3.5,2025-07-14,2025-08-25,3,Sunday
504,1-Weekday,7-Culver Boulevard,outbound,734,Culver Blvd/Inglewood Blvd,8.9,15.1,2025-07-14,2025-08-25,7,Weekday
313,1-Weekday,4-Jefferson Boulevard,Inbound,424,ObamaBlvd/KalsmanDr,0.2,2.2,2025-07-14,2025-08-25,4,Weekday


In [10]:
def normalize_day_type(
    df,
    day_type_col,
    day_of_week_col
):
    """
    Reclassifies 'Holiday' rows into Weekday/Saturday/Sunday
    based on the day of week.
    """
    return df.apply(
        lambda row: (
            'Weekday'
            if row[day_type_col] == 'Holiday'
            and row[day_of_week_col] in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            else (
                'Saturday'
                if row[day_type_col] == 'Holiday'
                and row[day_of_week_col] == 'Saturday'
                else (
                    'Sunday'
                    if row[day_type_col] == 'Holiday'
                    and row[day_of_week_col] == 'Sunday'
                    else row[day_type_col]
                )
            )
        ),
        axis=1
    )

In [11]:
# Function to count number of days of a Day_Type in a service period
def count_day_type_days(start, end, day_type):
    dates = pd.date_range(start, end)
    day_type_upper = day_type.upper()
    if day_type_upper == 'WEEKDAY':
        return sum(d.weekday() < 5 for d in dates)  
    elif day_type_upper == 'SATURDAY':
        return sum(d.weekday() == 5 for d in dates)
    elif day_type_upper == 'SUNDAY':
        return sum(d.weekday() == 6 for d in dates)
    else:
        return 0

In [42]:
def add_day_type(df, date_col, output_col='Day Type'):
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [12]:
bart_ridership['Day Type'] = normalize_day_type(
    bart_ridership,
    day_type_col='Day Type',
    day_of_week_col='Day of Week'
)

bart_ridership['exposure'] = bart_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

bart_ridership.rename(columns={
    'Station': 'Stop',
    'Station Name': 'Stop Name',
    'Entries': 'Boardings'
}, inplace=True)

In [13]:
# Compute exposure: number of days of that Day_Type in the period
big_blue_bus_ridership['exposure'] = big_blue_bus_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['SERVICE_DAY']),
    axis=1
)

# Compute total boardings for the period
big_blue_bus_ridership['Boardings'] = big_blue_bus_ridership['AVERAGE_DAILY_BOARDINGS'] * big_blue_bus_ridership['exposure']

# Rename columns
big_blue_bus_ridership.rename(columns={
    'SERVICE_DAY': 'Day_Type',
    'STOP_ID': 'Stop',
    'STOP_NAME': 'Stop Name'
}, inplace=True)


For Caltrain Ridership, dropping "Holiday" data since we just have average ridership for the month. 

In [14]:
caltrain_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Month               2520 non-null   object        
 1   Origin Station      2520 non-null   object        
 2   Caltrain Ridership  2520 non-null   float64       
 3   Date Type           2520 non-null   object        
 4   Average Ridership   2520 non-null   float64       
 5   start_date          2520 non-null   datetime64[ns]
 6   end_date            2520 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), object(3)
memory usage: 137.9+ KB


In [15]:
caltrain_ridership = caltrain_ridership[caltrain_ridership['Date Type'] != 'Holiday']


# Compute exposure: number of days of that Day_Type in the period
caltrain_ridership['exposure'] = caltrain_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Date Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
caltrain_ridership['Boardings'] = caltrain_ridership['Average Ridership'] * caltrain_ridership['exposure']

In [16]:
caltrain_ridership.sample(5)

,Month,Origin Station,Caltrain Ridership,Date Type,Average Ridership,start_date,end_date,exposure,Boardings
313,September 2024,Menlo Park,19735.381462,Weekday,718.464884,2024-09-01,2024-09-30,21,15087.762565
1865,November 2023,Burlingame,9688.468937,Sunday,117.775351,2023-11-01,2023-11-30,4,471.101402
152,February 2025,Belmont,14458.729323,Weekday,645.334618,2025-02-01,2025-02-28,20,12906.692361
905,October 2024,Burlingame,14827.718623,Saturday,348.693705,2024-10-01,2024-10-31,4,1394.774819
208,January 2025,Sunnyvale,39466.151403,Weekday,1573.436946,2025-01-01,2025-01-31,23,36189.049749


In [17]:
culver_citybus_ridership.sample(5)

,Day Of Week,Route,Direction,Stop ID,Stop Name,AVG On,AVG Off,start_date,end_date,route_id,day_type
903,3-Sunday,1-Washington Boulevard,Inbound,137,Washington Blvd/Cattaraugus Ave,1.3,10.1,2025-07-14,2025-08-25,1,Sunday
553,1-Weekday,Rapid 6-Sepulveda Boulevard,outbound,616,Sepulveda Blvd/76th St,0.0,1.0,2025-07-14,2025-08-25,Rapid 6,Weekday
207,1-Weekday,3-Crosstown,Inbound,329,Motor Ave/Woodbine St,5.7,9.4,2025-07-14,2025-08-25,3,Weekday
393,1-Weekday,6-Sepulveda Boulevard,Inbound,661,Sepulveda Blvd/Palms Blvd,3.0,1.0,2025-07-14,2025-08-25,6,Weekday
1118,3-Sunday,6-Sepulveda Boulevard,Inbound,911,Westwood Blvd/Kinross Ave,2.5,32.4,2025-07-14,2025-08-25,6,Sunday


In [18]:
group_cols = ['Day Of Week', 'Route', 'Stop ID', 'Stop Name', 
              'route_id', 'start_date', 'end_date', 'day_type']

# Sum AVG On and AVG Off
total_ridership_culver = culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg({
    'AVG On': 'sum'
})

Culver city : Missing records for either inbound or outbound for certain stops. For now, we are considering no service/no stop for that direction.

In [19]:
total_ridership_culver.head(5)

,Day Of Week,Route,Stop ID,Stop Name,route_id,start_date,end_date,day_type,AVG On
0,1-Weekday,1-Washington Boulevard,101,WindwardAve/MainSt,1,2025-07-14,2025-08-25,Weekday,111.2
1,1-Weekday,1-Washington Boulevard,102,Pacific Ave/N Venice Blvd,1,2025-07-14,2025-08-25,Weekday,31.7
2,1-Weekday,1-Washington Boulevard,103,Washington Blvd/Pacific Ave,1,2025-07-14,2025-08-25,Weekday,84.2
3,1-Weekday,1-Washington Boulevard,104,Washington Blvd/Via Dolce,1,2025-07-14,2025-08-25,Weekday,39.4
4,1-Weekday,1-Washington Boulevard,105,Washington Blvd/Via Marina,1,2025-07-14,2025-08-25,Weekday,42.4


In [20]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_culver['exposure'] = total_ridership_culver.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['day_type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_culver['Boardings'] = total_ridership_culver['AVG On'] * total_ridership_culver['exposure']

In [21]:
foothill_transit_ridership['day_type'].unique()

array(['weekday', 'holiday', 'weekend'], dtype=object)

In [22]:
foothill_transit_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694858 entries, 0 to 694857
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              694858 non-null  datetime64[ns]
 1   route_short_name  694858 non-null  int64         
 2   direction         694858 non-null  object        
 3   stop_code         694858 non-null  int64         
 4   stop_lat          694858 non-null  float64       
 5   stop_lon          694858 non-null  float64       
 6   boardings         694858 non-null  int64         
 7   alightings        694858 non-null  int64         
 8   start_date        694858 non-null  datetime64[ns]
 9   end_date          694858 non-null  datetime64[ns]
 10  day_type          694858 non-null  object        
dtypes: datetime64[ns](3), float64(2), int64(4), object(2)
memory usage: 58.3+ MB


In [ ]:
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col='date')

In [24]:
group_cols = ['date', 'route_short_name', 'stop_code', 'stop_lat', 'stop_lon', 
              'start_date', 'end_date', 'day_type', 'Day Type']

# Sum AVG On and AVG Off
total_ridership_foothill = foothill_transit_ridership.groupby(
    group_cols, as_index=False).agg({
    'boardings': 'sum'
})

In [25]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_foothill['exposure'] = total_ridership_foothill.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_foothill['Boardings'] = total_ridership_foothill['boardings'] * total_ridership_foothill['exposure']

In [29]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend


bart_ridership, caltrain, fresno_area_express_ridership no route information

In [40]:
fresno_area_express_ridership['day_type'].unique()

array(['weekend', 'holiday', 'weekday'], dtype=object)

In [52]:
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col='Date')

# Compute exposure: number of days of that Day_Type in the period
fresno_area_express_ridership['exposure'] = fresno_area_express_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
fresno_area_express_ridership['Boardings'] = fresno_area_express_ridership['ProjectedBoarding'] * fresno_area_express_ridership['exposure']

In [54]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type,Day Type,exposure,Boardings
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend,Sunday,1,44.691729
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend,Sunday,1,7.000000
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend,Sunday,1,20.000000
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend,Sunday,1,79.000000
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend,Sunday,1,63.000000


In [56]:
gold_coast_transit_ridership.head(5)

,day_of_week,route,direction,stop_id,unknown,stop_name,total_on,total_off,total_activity,cumulative_load,lat,lon,start_date,end_date
0,Weekday,1A,North,4THBST2,19,4th & B St,2,61,62,97,34.198975,-119.179621,2025-05-01,2025-05-31
1,Weekday,1A,South,4THBST1,1,4th & B St,70,2,72,188,34.199066,-119.179574,2025-05-01,2025-05-31
2,Weekday,1B,North,4THBST2,21,4th & B St,1,56,57,115,34.198975,-119.179621,2025-05-01,2025-05-31
3,Weekday,1B,South,4THBST1,1,4th & B St,63,2,65,183,34.199066,-119.179574,2025-05-01,2025-05-31
4,Weekday,1B,South,BAR5TH1,16,Bard & 5th,2,6,7,132,34.161169,-119.194368,2025-05-01,2025-05-31


In [58]:
gold_coast_transit_ridership['direction'].unique()

array(['North', 'South', 'Loop', 'East', 'West', 'PM', 'AM', 'A'],
      dtype=object)